In [5]:
import os
import json
import requests
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env
load_dotenv(dotenv_path='/content/claves.env')

# Configuración
API_KEY = os.getenv("API_KEY")
print(f"API_KEY: {API_KEY}")
CITY = os.getenv("CITY", "Buenos Aires")
OUTPUT_FILE = "data_extracted.json"

# API Pública: OpenWeatherMap (o adaptar URL a JSONPlaceholder / PokeAPI)
URL = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric&lang=es"

def extract_and_save_data():
    if not API_KEY:
        print("Error: No se encontró la API_KEY en el archivo .env")
        return

    try:
        print(f"Obteniendo datos de la API para: {CITY}...")
        response = requests.get(URL, timeout=10)

        # Validación explícita de Status Code
        response.raise_for_status()

        data = response.json()

        # Transformación básica: Filtrado de datos relevantes
        clean_data = {
            "ciudad": data.get("name"),
            "pais": data.get("sys", {}).get("country"),
            "temperatura_actual_c": data.get("main", {}).get("temp"),
            "sensacion_termica_c": data.get("main", {}).get("feels_like"),
            "humedad_porcentaje": data.get("main", {}).get("humidity"),
            "condicion": data.get("weather", [{}])[0].get("description")
        }

        # Persistencia de datos en archivo JSON
        with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
            json.dump(clean_data, file, ensure_ascii=False, indent=4)

        print(f"¡Éxito! Datos almacenados correctamente en '{OUTPUT_FILE}'.")

    except requests.exceptions.HTTPError as http_err:
        print(f"Error HTTP ocurrido: {http_err} (Código: {response.status_code})")
    except requests.exceptions.ConnectionError:
        print("Error de conexión: No se pudo conectar con la API. Revisa tu acceso a internet.")
    except requests.exceptions.Timeout:
        print("Error de tiempo de espera: La API tardó demasiado en responder.")
    except requests.exceptions.RequestException as err:
        print(f"Ocurrió un error inesperado al realizar la petición: {err}")
    except Exception as e:
        print(f"Error en el procesamiento o guardado de datos: {e}")

if __name__ == "__main__":
    extract_and_save_data()

API_KEY: 68ffe461880070f9d256ef8fe6bdd12c
Obteniendo datos de la API para: Buenos Aires...
¡Éxito! Datos almacenados correctamente en 'data_extracted.json'.
